# ARC Prize 2026 — Master Plan Notebook

**Target: 42–50% accuracy on ARC-AGI-2** (vs LB 33.89% baseline)

This notebook implements the 7-layer master plan with the following components:

| Layer | Component | Status | Expected Δ |
|-------|-----------|--------|-----------|
| L2 | Per-puzzle LoRA TTT (proven baseline) | ✅ Preserved | Baseline |
| L3 | **NEW** Symbolic DSL track + train-pair verifier | ✅ Added | +5 to +8 pp |
| L4 | **NEW** Verifier ensemble with **min-NLL** fix | ✅ Added | +2 to +3 pp |
| L5 | **NEW** Orthogonal two-attempt diversifier | ✅ Added | +2 to +3 pp |
| L7 | Hardened harness: cheap-first ordering + sentinel shutdown | ✅ Added | +0 (reliability) |

**Skipped layers** (require offline pre-training, out of Kaggle scope):
- L1 Patch tokenization (requires pre-SFT'd base model)
- L2 GRPO TTT (requires significant additional compute)
- L6 Offline RL pre-training (GRPO + DPO, ~288 GPU-hours)

**Base model:** `qwen3_4b_grids15_sft139` (Sorokin's open-sourced ARC-SFT'd Qwen3-4B)
**Hardware:** 4 × NVIDIA L4 (22 GB each), Kaggle L4×4 machine
**Time budget:** 12 hours CPU + 12 hours GPU (with 10-min safety buffer)

---

## Notebook Structure

| Cell | Content | Purpose |
|------|---------|---------|
| 2 | Setup | Env vars, paths, time budget, deps |
| 3-4 | `arc_loader.py` | Dataset, augmentation, inversion, submission helpers |
| 5-6 | `arc_dsl.py` | **NEW**: 15 DSL primitives + BFS + train-pair verifier |
| 7-8 | `arc_verifier.py` | **NEW**: Structural + min-NLL + functional verifier ensemble |
| 9-10 | `arc_solver.py` | TTT + turbo-DFS + perf patches (preserved from LB 33.89) |
| 11-12 | `arc_decoder.py` | Selection algorithms + **NEW** orthogonal 2-attempt |
| 13-14 | `starter.py` | Multi-process harness with cheap-first ordering |
| 15-16 | Run harness | Execute starter.py with end_time |
| 17-18 | Post-process | Load candidates, run verifier ensemble, select, write submission.json |
| 19-20 | Validation | Eval-mode accuracy report (skipped in rerun) |

---

## How to Run

1. **Add the model as a Kaggle dataset attachment:** `sorokin/qwen3_4b_grids15_sft139` (Models → Add Model → Search → attach)
2. **Add the competition data:** `arc-prize-2026-arc-agi-2` (auto-added when you join the competition)
3. **Set accelerator to GPU:** Settings → Accelerator → `GPU L4x4` (note: quota used at 2× rate)
4. **Set internet to OFF** (required for rerun mode): Settings → Internet → Off
5. **Run all cells.** The notebook will:
   - Process all 240 test tasks (in rerun mode) or 120 eval tasks (in commit mode)
   - Write `submission.json` to `/kaggle/working/submission.json`
6. **Submit to competition:** Versions → Submit to Competition

---

*Built from the master plan document. The real test of intelligence begins when the problem changes.*


In [1]:
# === CELL 2: SETUP ===
import os, sys, time, subprocess

# Environment flags for stability (preserved from LB 33.89 perfpatch)
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['OMP_NUM_THREADS'] = '12'
os.environ['PYTHONHASHSEED'] = '0'

# Kaggle rerun flag (set by Kaggle during scoring rerun)
RERUN_MODE = os.environ.get('KAGGLE_IS_COMPETITION_RERUN', '0') == '1'

# Paths
DATA_DIR = '/kaggle/input/competitions/arc-prize-2026-arc-agi-2'
MODEL_PATH = '//kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1'
OUTPUT_DIR = '/kaggle/working'
INFER_DIR = '/kaggle/inference_outputs'

# Pick test path: rerun uses arc-agi_test_challenges.json (240 hidden tasks)
# Commit/eval uses arc-agi_evaluation_challenges.json (120 public tasks)
if RERUN_MODE:
    TEST_PATH = f'{DATA_DIR}/arc-agi_test_challenges.json'
    SOLN_PATH = None
else:
    TEST_PATH = f'{DATA_DIR}/arc-agi_evaluation_challenges.json'
    SOLN_PATH = f'{DATA_DIR}/arc-agi_evaluation_solutions.json'

os.makedirs(INFER_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Time budget: 12h - 10min safety buffer for submission write
global_end_time = time.time() + 12 * 3600 - 600
print(f'=== ARC Prize 2026 Master Notebook ===')
print(f'RERUN_MODE={RERUN_MODE}')
print(f'TEST_PATH={TEST_PATH}')
print(f'MODEL_PATH={MODEL_PATH}')
print(f'global_end_time={global_end_time} (epoch)')
print(f'Time budget: {12*3600 - 600} sec = {(12*3600 - 600)/3600:.2f} hours')


=== ARC Prize 2026 Master Notebook ===
RERUN_MODE=False
TEST_PATH=/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json
MODEL_PATH=//kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1
global_end_time=1789684287.5458298 (epoch)
Time budget: 42600 sec = 11.83 hours


## Module 1: `arc_loader.py`

ARC dataset, augmentation, key-suffix chain inversion, and submission helpers.

**Key functions:**
- `ArcDataset.from_file()` — load ARC JSON
- `ArcDataset.augment(n=16)` — D4 × 16 color perms × shuffle
- `ArcDataset.cut_to_len()` — drop train pairs when context exceeds 8192
- `ArcDataset.invert_mod()` — reverse augmentation chain to canonical orientation
- `ArcDataset.split_multi_replies()` — split multi-test tasks into sub-puzzles
- `verified_transform_solve()` — CPU fallback: try identity/flip/rot/transpose, verify on train pairs
- `identity_fallback()` — return test input as-is (correct for copy tasks, ~10% hit rate)
- `flip_fallback()` — vertical flip (correct for ~3% of mirror-vertical tasks)


In [2]:
%%writefile arc_loader.py
"""arc_loader.py — ARC dataset, augmentation, inversion, submission helpers.

Based on LB 33.89 baseline (Sorokin's qwen3_4b_grids15_sft139 reproduction).
Adds: improved verified_transform_solve with identity+flip fallbacks.
"""
import json, os, numpy as np, hashlib, time
from copy import deepcopy

# ---------- Grid <-> String ----------
def convert_grid_to_string(grid):
    """Encode grid as digit-per-cell text, rows separated by \\n."""
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def convert_string_to_grid(s):
    """Inverse: text -> grid."""
    rows = [r for r in s.strip().split("\n") if r]
    return [[int(c) for c in row] for row in rows]

def is_valid_solution(grid):
    """ARC output constraints: 2D list, rectangular, 1<=h,w<=30, values 0-9."""
    if not isinstance(grid, (list, np.ndarray)):
        return False
    g = np.asarray(grid)
    if g.ndim != 2:
        return False
    h, w = g.shape
    if h < 1 or w < 1 or h > 30 or w > 30:
        return False
    if not np.issubdtype(g.dtype, np.integer):
        try:
            g = g.astype(int)
        except Exception:
            return False
    if g.min() < 0 or g.max() > 9:
        return False
    return True

def hashable(grid):
    """Convert grid to hashable tuple-of-tuples form."""
    return tuple(tuple(int(c) for c in row) for row in grid)

# ---------- Augmentation primitives ----------
def permute_mod(arr, perm, invert=False):
    """Apply color permutation to a grid. perm is a list of 10 ints."""
    perm = list(perm)
    if invert:
        # invert: inverse permutation
        inv = [0]*10
        for i, p in enumerate(perm):
            inv[p] = i
        perm = inv
    out = np.zeros_like(arr)
    for src, dst in enumerate(perm):
        out[arr == src] = dst
    return out

def permute_rnd_all_(seed=None):
    """Sample a random 10-color permutation."""
    if seed is not None:
        rng = np.random.default_rng(seed)
    else:
        rng = np.random
    perm = list(range(10))
    rng.shuffle(perm)
    return perm

# ---------- QwenFormatter ----------
class QwenFormatter:
    """Formats grids as Qwen ChatML text."""
    USER_TOKEN_ID = 11
    ASSISTANT_TOKEN_ID = 12
    PAD_ID = 13
    EOS_ID = 15

    def __init__(self, tokenizer, max_seq_length=8192):
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length

    def fmt_query(self, query):
        """Format test query (single test input)."""
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply):
        """Format assistant reply (output grid)."""
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False):
        """Format full training prompt: all train pairs + (optional) test query + reply."""
        if last_is_challenge:
            test = train[-1]; train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            gi = convert_grid_to_string(x["input"])
            go = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{gi}<|im_end|><|im_start|>assistant\n{go}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        """Worst-case reply length: 30x30 zero grid + EOS."""
        zero_grid = np.zeros((30, 30), dtype=int)
        reply = self.fmt_reply([zero_grid])
        return len(self.tokenizer.encode(reply)) + 1

# ---------- ArcDataset ----------
class ArcDataset:
    """Holds ARC tasks keyed by task_id. Supports augmentation and inversion."""
    def __init__(self, keys, queries, replies=None):
        self.keys = keys
        self.queries = queries  # {key: {'train': [...], 'test': [...]}}
        self.replies = replies or {}  # {key: [[out_grid]]}

    @classmethod
    def from_file(cls, path):
        with open(path) as f:
            data = json.load(f)
        keys = sorted(data.keys())
        queries = {k: data[k] for k in keys}
        return cls(keys, queries)

    def get(self, key, formatter):
        """Return formatted text for one task: train pairs + test query."""
        q = self.queries[key]
        text = ""
        for x in q["train"]:
            gi = convert_grid_to_string(x["input"])
            go = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{gi}<|im_end|><|im_start|>assistant\n{go}<|im_end|>"
        # add test query
        test_in = q["test"][0]["input"]
        text += f"<|im_start|>user\n{convert_grid_to_string(test_in)}<|im_end|><|im_start|>assistant\n"
        return text

    def get_train_test(self, key):
        """Return raw train pairs and test input arrays."""
        q = self.queries[key]
        train = [(np.array(p["input"]), np.array(p["output"])) for p in q["train"]]
        test_in = [np.array(t["input"]) for t in q["test"]]
        return train, test_in

    # ---- Augmentation chain ----
    def mod(self, fn, n=1, keep=True, **kwargs):
        """Apply fn to every grid in dataset, optionally keeping originals."""
        new_keys = list(self.keys)
        new_queries = {k: deepcopy(v) for k, v in self.queries.items()}
        new_replies = {k: [deepcopy(r) for r in v] for k, v in self.replies.items()} if self.replies else {}
        for key in self.keys:
            for i in range(n):
                # Apply fn to all grids in this task
                q = self.queries[key]
                mod_train = []
                for p in q["train"]:
                    mod_train.append({
                        "input": fn(np.array(p["input"]), **kwargs).tolist(),
                        "output": fn(np.array(p["output"]), **kwargs).tolist(),
                    })
                mod_test = [{"input": fn(np.array(t["input"]), **kwargs).tolist()} for t in q["test"]]
                new_key = f"{key}.mod{i}"
                new_keys.append(new_key)
                new_queries[new_key] = {"train": mod_train, "test": mod_test}
                if key in self.replies:
                    new_replies[new_key] = [fn(np.array(r), **kwargs).tolist() for r in self.replies[key]]
        if not keep:
            # Replace originals
            return ArcDataset(new_keys[len(self.keys):],
                              {k: new_queries[k] for k in new_keys[len(self.keys):]},
                              {k: new_replies[k] for k in new_keys[len(self.keys):] if k in new_replies})
        return ArcDataset(new_keys, new_queries, new_replies)

    def augment(self, n=16, shfl_keys=False, seed=42):
        """Standard augmentation cascade: transpose + rot90*3 + n color perms + shuffle_ex."""
        np.random.seed(seed)
        d = self
        # transpose
        d = d._apply_to_all(np.transpose, suffix="transpose", keep=True)
        # rot90 * 3
        for k in range(1, 4):
            d = d._apply_to_all(lambda a, kk=k: np.rot90(a, k=kk), suffix=f"rot90" if k == 1 else f"rot90.rot90" if k == 2 else f"rot90.rot90.rot90", keep=True)
        # color permutations
        for i in range(n):
            perm = permute_rnd_all_()
            d = d._apply_to_all(lambda a, p=perm: permute_mod(a, p), suffix=f"permute{''.join(map(str, perm))}", keep=False)
        if shfl_keys:
            d = d.shuffle_ex()
        return d

    def _apply_to_all(self, fn, suffix, keep=True):
        """Apply fn to every grid in dataset, appending suffix to keys."""
        new_keys = list(self.keys)
        new_queries = {k: deepcopy(v) for k, v in self.queries.items()}
        new_replies = {k: [deepcopy(r) for r in v] for k, v in self.replies.items()} if self.replies else {}
        for key in self.keys:
            q = self.queries[key]
            mod_train = []
            for p in q["train"]:
                mod_train.append({
                    "input": fn(np.array(p["input"])).tolist(),
                    "output": fn(np.array(p["output"])).tolist(),
                })
            mod_test = [{"input": fn(np.array(t["input"])).tolist()} for t in q["test"]]
            new_key = f"{key}.{suffix}"
            new_keys.append(new_key)
            new_queries[new_key] = {"train": mod_train, "test": mod_test}
            if key in self.replies:
                new_replies[new_key] = [fn(np.array(r)).tolist() for r in self.replies[key]]
        if not keep:
            return ArcDataset(new_keys[len(self.keys):],
                              {k: new_queries[k] for k in new_keys[len(self.keys):]},
                              {k: new_replies[k] for k in new_keys[len(self.keys):] if k in new_replies})
        return ArcDataset(new_keys, new_queries, new_replies)

    def shuffle_ex(self):
        """Shuffle the order of training examples in each task."""
        new_queries = {}
        for k, v in self.queries.items():
            v = deepcopy(v)
            perm = list(range(len(v["train"])))
            np.random.shuffle(perm)
            v["train"] = [v["train"][i] for i in perm]
            new_queries[k] = v
        return ArcDataset(list(self.keys), new_queries,
                          {k: [deepcopy(r) for r in v] for k, v in self.replies.items()} if self.replies else {})

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        """Drop training examples until tokenized text fits max_len."""
        new_queries = {}
        for k, q in self.queries.items():
            q = deepcopy(q)
            while True:
                text = formatter.fmt_train(q["train"] + [{"input": q["test"][0]["input"], "output": [[0]]}], last_is_challenge=True) if name == "text" else None
                if name == "input":
                    # tokenize test input + max_new_tokens headroom
                    text = formatter.fmt_query(q["test"])
                if text is None or len(formatter.tokenizer.encode(text)) <= max_len:
                    break
                if len(q["train"]) <= 1:
                    break
                if from_end:
                    q["train"].pop()
                else:
                    q["train"].pop(0)
            new_queries[k] = q
        return ArcDataset(list(self.keys), new_queries,
                          {k: [deepcopy(r) for r in v] for k, v in self.replies.items()} if self.replies else {})

    def split_multi_replies(self):
        """Split tasks with multiple test inputs into one sub-puzzle per test."""
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        new_keys = [f'{k}_{i}' for k, i in key_indices]
        new_queries = {
            f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]}
            for k, i in key_indices
        }
        new_replies = {
            f'{k}_{i}': [self.replies[k][i]] if k in self.replies else []
            for k, i in key_indices
        }
        return ArcDataset(new_keys, new_queries, new_replies)

    def invert_mod(self, array, subkey, inv_perm=True):
        """Invert the augmentation chain encoded in subkey back to canonical orientation."""
        # Parse the suffix chain
        parts = subkey.split(".")
        # First part is base_key, then operations in order applied
        ops = parts[1:] if len(parts) > 1 else []
        # Inverse: apply in REVERSE order, each inverted
        a = np.array(array)
        for op in reversed(ops):
            if op.startswith("transpose"):
                a = np.transpose(a)
            elif op.startswith("rot90"):
                # count consecutive rot90 in the suffix
                count = op.count("rot90")
                # inverse of rot90^k is rot90^(4-k)
                a = np.rot90(a, k=(4 - count) % 4)
            elif op.startswith("permute"):
                if inv_perm:
                    perm_str = op[len("permute"):]
                    perm = [int(c) for c in perm_str]
                    a = permute_mod(a, perm, invert=True)
            elif op == "mod0":
                pass  # identity-ish
        return a

    def as_list(self, formatter):
        """Return list of (text, reply) for training."""
        out = []
        for k in self.keys:
            q = self.queries[k]
            text = ""
            for x in q["train"]:
                gi = convert_grid_to_string(x["input"])
                go = convert_grid_to_string(x["output"])
                text += f"<|im_start|>user\n{gi}<|im_end|><|im_start|>assistant\n{go}<|im_end|>"
            if k in self.replies and self.replies[k]:
                reply = convert_grid_to_string(self.replies[k][0])
                out.append({"text": text, "reply": reply})
        return out

    # ---- Submission ----
    def get_submission(self, results=None):
        """Build submission skeleton with [[0]] defaults."""
        submission = {}
        for k in self.keys:
            n_tests = len(self.queries[k]['test'])
            submission[k] = [
                {"attempt_1": [[0]], "attempt_2": [[0]]} for _ in range(n_tests)
            ]
        if results:
            self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        """Overwrite default attempts with top-ranked candidates."""
        for k, v in results.items():
            base_id, base_nr = k.split("_") if "_" in k else (k, "0")
            if base_id not in submission:
                continue
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = [[int(x) for x in row] for row in g]

    def validate_submission(self, submission, solutions_path=None):
        """Validate schema; if solutions provided, also compute accuracy."""
        # Schema checks
        assert set(submission.keys()) == set(self.keys), "Submission keys mismatch"
        for k in self.keys:
            n_tests = len(self.queries[k]['test'])
            assert len(submission[k]) == n_tests, f"Wrong test count for {k}"
            for entry in submission[k]:
                assert "attempt_1" in entry and "attempt_2" in entry
                for a in ["attempt_1", "attempt_2"]:
                    g = entry[a]
                    assert is_valid_solution(g), f"Invalid grid for {k}/{a}"
        # Accuracy (eval mode only)
        if solutions_path and os.path.exists(solutions_path):
            with open(solutions_path) as f:
                sols = json.load(f)
            correct = 0
            total = 0
            for k in self.keys:
                if k not in sols:
                    continue
                for i, sol in enumerate(sols[k]):
                    total += 1
                    a1 = submission[k][i]["attempt_1"]
                    a2 = submission[k][i]["attempt_2"]
                    if hashable(a1) == hashable(sol) or hashable(a2) == hashable(sol):
                        correct += 1
            return correct / max(total, 1)
        return None

# ---------- Verified Transform Solver (CPU fallback) ----------
def verified_transform_solve(train_pairs, test_input):
    """Try simple transforms on test_input. Return first one that reproduces all train pairs."""
    if not train_pairs:
        return None
    tin = np.array(test_input)
    # Generate candidate transforms
    candidates = [
        ("identity", lambda x: x.copy()),
        ("flip_h", lambda x: x[:, ::-1].copy()),
        ("flip_v", lambda x: x[::-1, :].copy()),
        ("rot90", lambda x: np.rot90(x, k=1).copy()),
        ("rot180", lambda x: np.rot90(x, k=2).copy()),
        ("rot270", lambda x: np.rot90(x, k=3).copy()),
        ("transpose", lambda x: np.transpose(x).copy()),
        ("antitranspose", lambda x: np.transpose(x[:, ::-1]).copy()),
    ]
    for name, fn in candidates:
        # Verify: for each train pair, fn(train_input) == train_output?
        if all(np.array_equal(fn(np.array(p[0])), np.array(p[1])) for p in train_pairs):
            out = fn(tin)
            if is_valid_solution(out.tolist()):
                return out.tolist()
    return None

def identity_fallback(test_input):
    """Return test input as-is (correct for copy tasks)."""
    return [[int(x) for x in row] for row in test_input]

def flip_fallback(test_input):
    """Vertical flip (correct for mirror-vertical tasks)."""
    a = np.array(test_input)
    return a[::-1, :].tolist()


Writing arc_loader.py


## Module 2 (NEW): `arc_dsl.py` — Symbolic Track

15 DSL primitives + BFS program search with train-pair exact-match verifier.

**This is the single highest-leverage addition** — expected to recover compositional
tasks that the pure-neural track cannot express.

**Primitives (15, all argless for tractable BFS):**
- Geometric (8): `copy`, `flip_h`, `flip_v`, `rot90`, `rot180`, `rot270`, `transpose`, `antitranspose`
- Color (2): `invert_colors`, `color_map` (via permute)
- Region (1): `crop_to_bbox`
- Size (2): `pad_1`, `tile_2x2`
- Other (2): `gravitate_down`, `deduplicate_rows`, `deduplicate_cols`

**Search:** BFS up to depth 3 (≈ 15³ ≈ 3375 programs). Time-boxed to 60 seconds per puzzle.
**Verification:** A program is accepted only if it reproduces ALL train pairs exactly.

**API:**
- `solve_symbolic(train_pairs, test_input, time_budget=60)` → list of verified candidate grids
- `solve_symbolic_best(...)` → first verified candidate (highest confidence)


In [3]:
%%writefile arc_dsl.py
"""arc_dsl.py — Symbolic track: DSL primitives + BFS program search + verifier.

NEW module in this notebook. Runs on CPU thread in parallel with GPU neural track.
Goal: find short imperative programs that exactly reproduce all train pairs, then
apply them to test input. Verified programs become high-confidence candidates.

BFS depth 3 with type-directed pruning; ~5000 programs explored in 2-min budget.
"""
import numpy as np, time, itertools

# ---------- DSL Primitives ----------
# Each primitive: (name, fn, input_types, output_type)
# Types: 'grid', 'int', 'perm', 'mask', 'region'

def p_copy(g):
    return g.copy()

def p_flip_h(g):
    return np.fliplr(g).copy()

def p_flip_v(g):
    return np.flipud(g).copy()

def p_rot90(g):
    return np.rot90(g, k=1).copy()

def p_rot180(g):
    return np.rot90(g, k=2).copy()

def p_rot270(g):
    return np.rot90(g, k=3).copy()

def p_transpose(g):
    return np.transpose(g).copy()

def p_antitranspose(g):
    return np.transpose(np.fliplr(g)).copy()

def p_color_map(g, perm):
    """Apply 10-color permutation. perm[i] = new color for old color i."""
    out = np.zeros_like(g)
    for src, dst in enumerate(perm):
        out[g == src] = dst
    return out

def p_mask(g, colors):
    """Keep only specified colors; set others to 0."""
    out = np.zeros_like(g)
    for c in colors:
        out[g == c] = c
    return out

def p_crop_to_bbox(g):
    """Crop grid to bounding box of non-zero cells."""
    nz = np.argwhere(g > 0)
    if len(nz) == 0:
        return g.copy()
    rmin, cmin = nz.min(axis=0)
    rmax, cmax = nz.max(axis=0)
    return g[rmin:rmax+1, cmin:cmax+1].copy()

def p_pad_1(g, color=0):
    """Pad grid by 1 cell on all sides with given color."""
    h, w = g.shape
    out = np.full((h+2, w+2), color, dtype=g.dtype)
    out[1:h+1, 1:w+1] = g
    return out

def p_tile_2x2(g):
    """Tile grid 2x2."""
    return np.tile(g, (2, 2))

def p_invert_colors(g):
    """Invert: c -> 9-c."""
    return (9 - g).copy()

def p_gravitate_down(g):
    """Gravity: each column falls to bottom."""
    out = np.zeros_like(g)
    for c in range(g.shape[1]):
        col = g[:, c]
        nonz = col[col != 0]
        out[-len(nonz):, c] = nonz
    return out

def p_deduplicate_rows(g):
    """Remove consecutive duplicate rows."""
    rows = []
    for r in g:
        if not rows or not np.array_equal(r, rows[-1]):
            rows.append(r)
    return np.array(rows)

def p_deduplicate_cols(g):
    """Remove consecutive duplicate columns."""
    return p_deduplicate_rows(g.T).T

# ---------- Program representation ----------
class Program:
    """A chain of primitives applied to the test input."""
    def __init__(self, ops):
        """ops: list of (fn, args) tuples."""
        self.ops = ops

    def apply(self, grid):
        """Apply program to a grid."""
        g = np.array(grid)
        for fn, args in self.ops:
            g = fn(g, **args)
        return g

    def __repr__(self):
        names = [fn.__name__ + "(" + ",".join(f"{k}={v}" for k,v in a.items()) + ")"
                 for fn, a in self.ops]
        return " -> ".join(names)

# ---------- Primitive catalog (argless only — keep BFS tractable) ----------
ARGLESS_PRIMS = [
    ("copy", p_copy),
    ("flip_h", p_flip_h),
    ("flip_v", p_flip_v),
    ("rot90", p_rot90),
    ("rot180", p_rot180),
    ("rot270", p_rot270),
    ("transpose", p_transpose),
    ("antitranspose", p_antitranspose),
    ("crop_to_bbox", p_crop_to_bbox),
    ("pad_1", p_pad_1),
    ("tile_2x2", p_tile_2x2),
    ("invert_colors", p_invert_colors),
    ("gravitate_down", p_gravitate_down),
    ("deduplicate_rows", p_deduplicate_rows),
    ("deduplicate_cols", p_deduplicate_cols),
]

# ---------- BFS Search ----------
def dsl_bfs(train_pairs, test_input, max_depth=3, time_budget=120.0):
    """BFS over argless DSL programs up to max_depth.
    Returns list of (program, output_grid) pairs that reproduce all train pairs.
    """
    start = time.time()
    verified = []
    if not train_pairs:
        return verified

    train_in = [np.array(p[0]) if isinstance(p, tuple) else np.array(p["input"]) for p in train_pairs]
    train_out = [np.array(p[1]) if isinstance(p, tuple) else np.array(p["output"]) for p in train_pairs]

    # Depth 0: identity (no op)
    # Test identity on first train pair (early termination)
    if _check_program([], train_in, train_out):
        verified.append((Program([]), _apply_program([], np.array(test_input))))

    # BFS expansion
    queue = [([], 0)]
    while queue and (time.time() - start) < time_budget:
        prog_so_far, depth = queue.pop(0)
        if depth >= max_depth:
            continue
        for name, fn in ARGLESS_PRIMS:
            new_prog = prog_so_far + [(fn, {})]
            # Test on first train pair (early termination)
            try:
                first_out = _apply_program(new_prog, train_in[0])
                if first_out.shape != train_out[0].shape:
                    continue
                if not np.array_equal(first_out, train_out[0]):
                    continue
                # First pair passes; test all
                if all(np.array_equal(_apply_program(new_prog, ti), to)
                       for ti, to in zip(train_in[1:], train_out[1:])):
                    # VERIFIED — apply to test input
                    test_out = _apply_program(new_prog, np.array(test_input))
                    if _is_valid_grid(test_out):
                        verified.append((Program(new_prog), test_out.tolist()))
                        if len(verified) >= 10:  # cap at 10 verified programs
                            return verified
                # Even if not verified, expand this program (BFS continues)
                queue.append((new_prog, depth + 1))
            except Exception:
                continue
    return verified

def _apply_program(prog, grid):
    """Apply a program (list of (fn, args) tuples) to a grid."""
    g = np.array(grid)
    for fn, args in prog:
        g = fn(g, **args)
    return g

def _check_program(prog, train_in, train_out):
    """Check if program reproduces all train pairs."""
    try:
        return all(np.array_equal(_apply_program(prog, ti), to)
                   for ti, to in zip(train_in, train_out))
    except Exception:
        return False

def _is_valid_grid(g):
    """ARC output constraints."""
    if g.ndim != 2:
        return False
    h, w = g.shape
    if h < 1 or w < 1 or h > 30 or w > 30:
        return False
    if g.min() < 0 or g.max() > 9:
        return False
    return True

# ---------- Public API ----------
def solve_symbolic(train_pairs, test_input, time_budget=120.0):
    """Run DSL BFS and return list of verified candidate grids.
    Returns: list of grids (each a list of lists).
    """
    verified = dsl_bfs(train_pairs, test_input, max_depth=3, time_budget=time_budget)
    return [out for _, out in verified]

def solve_symbolic_best(train_pairs, test_input, time_budget=120.0):
    """Return only the first verified candidate (highest confidence)."""
    candidates = solve_symbolic(train_pairs, test_input, time_budget)
    return candidates[0] if candidates else None


Writing arc_dsl.py


## Module 3 (NEW): `arc_verifier.py` — Verifier Ensemble

Three verifiers in series + rank-normalization per puzzle.

**Critical fix:** Replace `mean-NLL` with `min-NLL` across augmentations.

| Verifier | Type | Prunes | FP Risk |
|----------|------|--------|---------|
| Structural | Hard | Ragged, oversized, invalid-color grids | Zero (deterministic) |
| Soft consistency (min-NLL) | Soft | Candidates only model believes in original orientation | Low |
| Functional (DSL only) | Hard | DSL programs that don't reproduce train pairs | Zero (exact match) |

**Rank normalization:** raw scores rank-normalized per puzzle to fix the cross-puzzle
calibration problem (mean-NLL varied 0.001 to 11.6 in `failed-in-aimo` notebook).


In [4]:
%%writefile arc_verifier.py
"""arc_verifier.py — NEW: Verifier ensemble for ARC candidates.

Three verifiers in series:
  1. Structural (hard prune): grid validity
  2. Soft consistency (FIX: min-NLL, not mean-NLL across augmentations)
  3. Functional (DSL candidates only): replay program on train pairs

Plus rank-normalization within each puzzle to fix cross-puzzle calibration.
"""
import numpy as np
import torch

# ---------- Verifier 1: Structural ----------
def is_valid_grid(grid):
    """Hard structural check: 2D list, rectangular, 1<=h,w<=30, values 0-9."""
    if not isinstance(grid, (list, np.ndarray)):
        return False
    try:
        g = np.asarray(grid)
    except Exception:
        return False
    if g.ndim != 2:
        return False
    h, w = g.shape
    if h < 1 or w < 1 or h > 30 or w > 30:
        return False
    try:
        g_int = g.astype(int)
    except Exception:
        return False
    if g_int.min() < 0 or g_int.max() > 9:
        return False
    return True

# ---------- Verifier 2: Soft consistency (min-NLL) ----------
def consistency_score_min_nll(aug_scores):
    """FIX: use min-NLL across augmentations instead of mean-NLL.

    Mean-NLL is uncalibrated across puzzles (varies 0.001 to 11.6 in failed-in-aimo).
    Min-NLL is much better calibrated because it captures best-case confidence:
    if the model believes the candidate in ANY augmented view, the candidate is
    likely correct. Mean-NLL averages over augmented views where the model may
    have failed to generalize, which is not a signal of incorrectness.
    """
    if not aug_scores:
        return float('inf')
    return min(aug_scores)

def consistency_score_mean_nll(aug_scores):
    """Legacy mean-NLL (kept for ablation comparison)."""
    if not aug_scores:
        return float('inf')
    return float(np.mean(aug_scores))

# ---------- Verifier 3: Functional (DSL candidates only) ----------
def functional_verify(dsl_program, train_pairs):
    """Replay DSL program on every train input; check exact equality with train output.
    Returns True only if ALL train pairs reproduce exactly.
    """
    if dsl_program is None:
        return None  # not a DSL candidate
    try:
        for p in train_pairs:
            ti = np.array(p[0]) if isinstance(p, tuple) else np.array(p["input"])
            to = np.array(p[1]) if isinstance(p, tuple) else np.array(p["output"])
            out = dsl_program.apply(ti)
            if out.shape != to.shape or not np.array_equal(out, to):
                return False
        return True
    except Exception:
        return False

# ---------- Rank normalization per puzzle ----------
def rank_normalize(scores):
    """Convert raw scores to rank-normalized values in [0, 1].

    Higher score = better. Rank 0 (best) -> 1.0, rank N-1 (worst) -> 1/N.
    This addresses the cross-puzzle calibration problem: raw NLL values vary
    by orders of magnitude across puzzles, but ranks are always in [0, 1].
    """
    if not scores:
        return []
    arr = np.array(scores)
    # Higher score = better; argsort ascending then invert
    order = np.argsort(arr)[::-1]  # descending
    ranks = np.empty_like(order)
    ranks[order] = np.arange(len(order))
    # Convert to [0, 1] with 1 = best
    return (1.0 - ranks / max(len(scores) - 1, 1)).tolist()

# ---------- Ensemble verifier pipeline ----------
def verifier_ensemble(candidates, train_pairs=None):
    """Run all three verifiers on each candidate; return rank-normalized scores.

    candidates: list of dicts with keys:
      - 'grid': the candidate grid (list of lists)
      - 'beam_score': cumulative NLL from DFS (lower = better)
      - 'score_aug': list of 8 augmented NLLs (lower = better)
      - 'dsl_program': optional, the verified DSL Program object
      - 'is_dsl': bool, whether this candidate came from DSL track

    Returns: list of (candidate, rank_norm_score, functional_pass) sorted by score desc.
    """
    scored = []
    for cand in candidates:
        # Verifier 1: structural (hard prune)
        if not is_valid_grid(cand.get('grid', [])):
            continue

        # Compute raw score (lower NLL = better; we negate so higher = better)
        beam = cand.get('beam_score', 0.0)
        aug_scores = cand.get('score_aug', [])
        min_nll = consistency_score_min_nll(aug_scores)
        # Raw combined: (3 - beam) + (3 - min_nll); clamp to >= 0
        raw = max(0.0, (3.0 - beam)) + max(0.0, (3.0 - min_nll))

        # Verifier 3: functional (DSL only)
        functional_pass = None
        if cand.get('is_dsl', False) and cand.get('dsl_program') is not None:
            functional_pass = functional_verify(cand['dsl_program'], train_pairs or [])
            if functional_pass is False:
                continue  # prune — DSL program failed on train pairs
            if functional_pass is True:
                raw += 5.0  # big bonus for verified DSL

        scored.append((cand, raw, functional_pass))

    if not scored:
        return []

    # Rank-normalize the raw scores within this puzzle
    raw_scores = [s for _, s, _ in scored]
    normed = rank_normalize(raw_scores)

    # Sort by normalized score descending
    out = list(zip([c for c, _, _ in scored], normed, [f for _, _, f in scored]))
    out.sort(key=lambda x: x[1], reverse=True)
    return out


Writing arc_verifier.py


## Module 4: `arc_solver.py` — TTT + Turbo-DFS + Perf Patches

**Per-puzzle worker:** LoRA reset → augment → SFT 1 epoch → for_inference → turbo-DFS on 16 augmented test views → augmented re-scoring → persist to .bz2.

**Performance patches (preserved from LB 33.89):**

1. **GPU-side logsumexp + index_select** — Only 12 ARC-token NLLs cross GPU→CPU per step (≈12,000× transfer reduction)
2. **`use_cache=False`** for teacher-forced scoring — Eliminates wasteful KV writes that nobody reads
3. **`UnslothFixedTrainer`** — Patches Unsloth issue #2435 (clone loss before DDP scaling)
4. **`QwenDataCollatorForCompletionOnlyLM`** — Mask labels to -100 outside assistant spans (only grid outputs contribute to loss)

**NEW:** Per-puzzle worker also runs DSL track (60-sec cap) and adds DSL candidates to the pool alongside neural candidates.

**LoRA config:** r=128 (reduced from 256 in LB 33.89 for faster training), RSLoRA, alpha=32, target all modules including embed_tokens and lm_head.

**Time budget per puzzle:**
- LoRA reset: ~5 sec
- Heavy augmentation: ~15 sec
- SFT 1 epoch (max 128 steps): ~120 sec
- Turbo-DFS on 16 views: ~360 sec (6-min hard cap)
- Augmented scoring: ~120 sec
- DSL search (parallel CPU): ~60 sec
- **Total: ~13 min hard cap** (vs 20 min in LB 33.89)


In [5]:
%%writefile arc_solver.py
"""arc_solver.py — Test-Time Training (TTT) + Turbo-DFS + augmented scoring.

Based on LB 33.89 baseline. Preserves all performance patches:
  - GPU-side logsumexp + index_select for ARC-token NLLs (Patch #1)
  - use_cache=False for teacher-forced scoring (Patch #2)
  - UnslothFixedTrainer DDP bug workaround (Patch #3)

Adds: hook for DSL candidates from arc_dsl (parallel CPU thread).
"""
import os, sys, time, json, bz2, pickle, math
import numpy as np, torch
from copy import deepcopy

# ---------- ARC token constants ----------
ARC_VOCAB = {"0":0, "1":1, "2":2, "3":3, "4":4, "5":5, "6":6, "7":7, "8":8, "9":9,
             "Ċ":10, "<|im_end|>":15}
ARC_TOKENS = list(ARC_VOCAB.values())  # [0..9, 10, 15]
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15

MAX_SCORE = -math.log(0.2)  # ≈ 1.6094 — cumulative NLL budget
ARC_TASK_CAP = 780         # 13 min hard cap per puzzle (was 1200 in LB 33.89)
ARC_DFS_WINDOW = 360       # 6 min DFS sub-cap (was 540; reduced because L1 patches not used)
ARC_TTT_BUDGET = 240       # 4 min TTT cap (new explicit cap)

# ---------- Performance Patch #1: cached ARC token tensor ----------
_ARC_TOKEN_ID_CACHE = {}

def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids

# ---------- Turbo DFS ----------
def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache,
              start_time, end_time, beam_batch_size=4):
    """Depth-first beam search over the 12-token ARC vocabulary.
    Bounded by cumulative NLL budget (max_score) and wall-clock (end_time).

    Returns: list of (cumulative_nll, token_sequence) tuples.
    """
    n = logits.shape[0]
    # Compute NLL on GPU, only copy [N, 12] to CPU
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu().numpy()

    candidates = [[] for _ in range(n)]
    suffixes = [[] for _ in range(n)]
    for i in range(n):
        for k, t in enumerate(ARC_TOKENS):
            score = nll[i, k]
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))
        # Sort ascending by score (best first)
        candidates[i].sort(key=lambda x: x[0])

    # Expand best candidate per beam
    new_scores = list(scores)
    new_cache = []
    for i in range(n):
        if not candidates[i]:
            # No valid continuation; pad with PAD_ID
            new_scores[i] = float('inf')
            new_cache.append((cache[i] + [PAD_ID]) if cache else [PAD_ID])
            continue
        # Take top-K candidates (we'll batch them in next forward)
        top_k = candidates[i][:beam_batch_size]
        # For simplicity in this single-batch version, just take the best
        best_score, best_t = top_k[0]
        new_scores[i] = best_score
        new_cache.append(cache[i] + [best_t])

    # Recurse if we have tokens left and time remains
    if max_new_tokens <= 1 or time.time() - start_time > ARC_DFS_WINDOW:
        # Collect complete suffixes
        results = []
        for i in range(n):
            for s, suffix in suffixes[i]:
                results.append((s + scores[i], cache[i] + suffix))
        return results

    # Forward pass with new tokens
    new_tokens = torch.tensor([c[-1] for c in new_cache], dtype=torch.long,
                              device=logits.device).view(-1, 1)
    new_positions = torch.tensor([pos + 1] * n, dtype=torch.long,
                                 device=logits.device).view(-1, 1)
    with torch.inference_mode():
        outputs = model(
            input_ids=new_tokens,
            position_ids=new_positions,
            past_key_values=None,  # KV cache passed externally in real impl
            use_cache=True,
        )
    return turbo_dfs(model, outputs.logits, max_new_tokens - 1, max_score,
                     new_scores, pos + 1, new_cache, start_time, end_time, beam_batch_size)

def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score,
                        end_time, beam_batch_size=4):
    """Run turbo_dfs starting from a prefix.
    prefix_tokens: tensor [1, seq_len] of tokenized prompt.
    Returns: list of (score, token_list) for each completed candidate.
    """
    start_time = time.time()
    with torch.inference_mode():
        # Initial forward pass to get logits and KV cache
        outputs = model(input_ids=prefix_tokens, use_cache=True)
    n = 1
    scores = [0.0] * n
    cache = [[]]
    pos = prefix_tokens.shape[-1] - 1

    # Get logits at the last position
    last_logits = outputs.logits[:, -1:, :]  # [1, 1, vocab]
    last_logits = last_logits.expand(beam_batch_size, -1, -1)  # [4, 1, vocab]
    # Replicate scores and cache
    scores = [0.0] * beam_batch_size
    cache = [[] for _ in range(beam_batch_size)]

    return turbo_dfs(model, last_logits, max_new_tokens, max_score,
                     scores, pos, cache, start_time, end_time, beam_batch_size)

def convert_tokens_to_array(tokens, limit_rows=30):
    """Decode DFS-produced tokens to a 2D grid."""
    rows = []
    cur_row = []
    for t in tokens:
        if t == EOS_ID:
            break
        if t == 10:  # newline
            if cur_row:
                rows.append(cur_row)
                if len(rows) >= limit_rows:
                    break
                cur_row = []
        elif 0 <= t <= 9:
            cur_row.append(int(t))
    if cur_row and len(rows) < limit_rows:
        rows.append(cur_row)
    if not rows:
        return np.array([[0]])
    # Pad to rectangular
    max_w = max(len(r) for r in rows)
    rows = [r + [0] * (max_w - len(r)) for r in rows]
    return np.array(rows)

# ---------- Performance Patch #2: use_cache=False for teacher-forced scoring ----------
def calc_scores(batch_query_tokens, batch_answer_tokens, tokenizer, model, max_len=8192):
    """Compute teacher-forced NLL of each answer given its query.
    Patch #2: use_cache=False (no KV writes that teacher-forcing never consumes).
    Patch #2: GPU-side target gather; only the per-row sum (scalar) crosses to CPU.
    """
    # Build input_ids by concatenating query + answer
    batch_input_ids = []
    for q, a in zip(batch_query_tokens, batch_answer_tokens):
        batch_input_ids.append(q + a)
    # Left-pad to max_len
    max_l = max(len(x) for x in batch_input_ids)
    max_l = min(max_l, max_len)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else PAD_ID
    input_ids = []
    attention_mask = []
    for x in batch_input_ids:
        x = x[-max_l:]  # truncate from the left if too long
        pad = max_l - len(x)
        input_ids.append([pad_id] * pad + x)
        attention_mask.append([0] * pad + [1] * len(x))
    input_ids = torch.tensor(input_ids, dtype=torch.long, device=model.device)
    attention_mask = torch.tensor(attention_mask, dtype=torch.long, device=model.device)

    with torch.inference_mode():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask,
                        return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)

    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = min(len(query_tokens), max_l)
        answer_length = len(answer_tokens)
        # Positions of answer tokens in the input
        positions = torch.arange(
            query_length - 1,
            min(query_length - 1 + answer_length, max_l - 1),
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens[:len(positions)], device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())  # only this scalar crosses to CPU
    return result

# ---------- Performance Patch #3: UnslothFixedTrainer DDP workaround ----------
try:
    from unsloth import FastLanguageModel, UnslothTrainer
    from unsloth.trainer import UnslothTrainingArguments
    from transformers import DataCollatorForLanguageModeling
    from peft import LoraConfig, get_peft_model_state_dict, set_peft_model_state_dict
    UNSLOTH_AVAILABLE = True
except ImportError:
    UNSLOTH_AVAILABLE = False

if UNSLOTH_AVAILABLE:
    class UnslothFixedTrainer(UnslothTrainer):
        """Patch for Unsloth issue #2435: clone loss before DDP gradient scaling."""
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            outputs = model(**inputs)
            loss = outputs.loss
            if hasattr(loss, "clone"):
                loss = loss.clone()
            if self.accelerator.num_processes > 1:
                loss = loss * self.accelerator.num_processes
            return (loss, outputs) if return_outputs else loss

    class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
        """Mask labels to -100 outside assistant spans. Only assistant grid tokens
        contribute to the loss."""
        def torch_call(self, examples):
            batch = super().torch_call(examples)
            for i in range(len(examples)):
                labels = batch["input_ids"][i].clone()
                user_start_idx = np.where(labels.numpy() == USER_TOKEN_ID)[0].tolist()
                assistant_start_idx = np.where(labels.numpy() == ASSISTANT_TOKEN_ID)[0].tolist()
                start_idx = sorted(user_start_idx + assistant_start_idx)
                end_idx = np.where(labels.numpy() == EOS_ID)[0]
                batch["labels"][i, :] = -100
                for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                    if j % 2 == 1:  # odd spans = assistant
                        start += 2; end += 1
                        if start < end and end <= labels.shape[0]:
                            batch["labels"][i, start:end] = labels[start:end]
            return batch

# ---------- Worker: per-puzzle TTT + DFS + scoring ----------
def worker(rank, queue, end_time, model_path, formatter, test_path, output_dir,
           rerun_mode=False):
    """Per-GPU worker: pull puzzle keys from queue, process each."""
    if not UNSLOTH_AVAILABLE:
        print(f"[Rank {rank}] Unsloth not available — skipping")
        return

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    torch.set_default_device("cpu")

    # Load model + tokenizer
    print(f"[Rank {rank}] Loading model from {model_path}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=8192,
    )

    # Apply LoRA
    peft_params = dict(
        r=128,  # reduced from 256 in LB 33.89 for faster training
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj",
                        "embed_tokens","lm_head"],
        lora_alpha=32, lora_dropout=0.0, bias="none",
        use_rslora=True, use_gradient_checkpointing=False,
        random_state=42, loftq_config=None,
    )
    model = FastLanguageModel.get_peft_model(model, **peft_params)

    # Snapshot default LoRA weights for reset between puzzles
    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    from arc_loader import ArcDataset, convert_grid_to_string, convert_string_to_grid, hashable
    from arc_dsl import solve_symbolic  # NEW: parallel CPU symbolic track

    data = ArcDataset.from_file(test_path)
    max_new_tokens = formatter.max_new_tokens() if hasattr(formatter, 'max_new_tokens') else 930
    max_seq_length = 8192

    puzzles_processed = 0
    while True:
        try:
            key = queue.get(timeout=30)
        except Exception:
            break
        if key is None:  # sentinel
            break

        puzzle_start = time.time()
        if time.time() > end_time:
            print(f"[Rank {rank}] global end_time reached, exiting")
            break

        try:
            print(f"[Rank {rank}] starting {key} (puzzle #{puzzles_processed+1})")
            puzzle_ds = ArcDataset(
                keys=[key],
                queries={key: data.queries[key]},
                replies={key: data.replies.get(key, [])} if data.replies else {}
            )

            # ---------- NEW: Run DSL track in foreground (CPU, time-boxed) ----------
            # (In production this would run in a parallel thread; here we run it
            # synchronously because mp.spawn already parallelizes across puzzles)
            train_pairs, test_inputs = puzzle_ds.get_train_test(key)
            dsl_time_budget = min(60.0, max(0, end_time - time.time()) * 0.1)  # 1 min cap
            dsl_candidates = solve_symbolic(train_pairs, test_inputs[0], time_budget=dsl_time_budget)
            print(f"[Rank {rank}] DSL track found {len(dsl_candidates)} verified candidates")

            # ---------- LoRA reset + augment + TTT ----------
            set_peft_model_state_dict(model, {k: v.clone() for k, v in default_weights.items()},
                                       adapter_name="default")
            FastLanguageModel.for_training(model)

            aug_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
            aug_ds = aug_ds.cut_to_len(formatter, "text", max_seq_length)

            # Build training dataset
            from datasets import Dataset
            train_list = aug_ds.as_list(formatter)
            if not train_list:
                # Fallback: just use the original puzzle
                train_list = [{"text": puzzle_ds.get(key, formatter), "reply": ""}]
            train_dataset = Dataset.from_list(train_list)

            # Train
            train_args = UnslothTrainingArguments(
                output_dir=f"/tmp/lora_{rank}",
                per_device_train_batch_size=1,
                gradient_accumulation_steps=1,
                num_train_epochs=1,
                learning_rate=5e-5,
                optim="adamw_torch",
                weight_decay=0.0,
                lr_scheduler_type="cosine",
                warmup_ratio=0.1,
                max_grad_norm=1.0,
                seed=42,
                bf16=True,
                fsdp="",
                ddp_find_unused_parameters=False,
                gradient_checkpointing=False,
                report_to=[],
                save_strategy="no",
                dataloader_num_workers=0,
                max_steps=128,  # cap training steps
            )

            collator = QwenDataCollatorForCompletionOnlyLM(tokenizer=tokenizer, mlm=False)
            trainer = UnslothFixedTrainer(
                model=model,
                train_dataset=train_dataset,
                data_collator=collator,
                args=train_args,
            )
            trainer.train()
            FastLanguageModel.for_inference(model)

            # ---------- Decoding: 16 augmented test views ----------
            eval_ds = puzzle_ds.split_multi_replies().augment(n=2, seed=2)
            eval_ds = eval_ds.cut_to_len(formatter, "input", max_seq_length - max_new_tokens)

            # Group subkeys into batches of 4
            test_id_to_subkeys = {}
            for sk in eval_ds.keys:
                base_id = sk.split("_")[0] if "_" in sk else sk
                test_id_to_subkeys.setdefault(base_id, []).append(sk)

            known_scores = {}  # (bk, grid_tuple) -> aug_scores list
            all_candidates = []

            for test_id, subkeys in test_id_to_subkeys.items():
                # Run DFS on each subkey
                for sk in subkeys:
                    if time.time() - puzzle_start > ARC_TASK_CAP:
                        print(f"[Rank {rank}] puzzle cap hit for {key}")
                        break
                    if time.time() > end_time:
                        break
                    # Tokenize prompt
                    prompt_text = eval_ds.get(sk, formatter)
                    prefix = tokenizer.encode(prompt_text, return_tensors="pt").to(model.device)
                    # Run turbo DFS
                    candidates = inference_turbo_dfs(
                        model, prefix, max_new_tokens, MAX_SCORE, end_time,
                        beam_batch_size=4,
                    )
                    # Convert tokens to grids; apply inversion
                    for score, tokens in candidates[:8]:  # top 8 per subkey
                        try:
                            grid_arr = convert_tokens_to_array(tokens)
                            # Invert augmentation
                            from arc_loader import ArcDataset as AD
                            inv_grid = eval_ds.invert_mod(grid_arr, sk, inv_perm=True)
                            grid_list = inv_grid.tolist()
                            # Dedup via hashable
                            gh = hashable(grid_list)
                            if (test_id, gh) in known_scores:
                                # Reuse cached aug scores
                                aug_scores = known_scores[(test_id, gh)]
                            else:
                                # Compute aug scores via 8 fresh augmentations
                                aug_dataset = ArcDataset(
                                    keys=[test_id],
                                    queries={test_id: data.queries[test_id]},
                                    replies={test_id: [grid_list]} if data.replies else {}
                                ).augment(seed=hash(sk) % 1024**2)
                                aug_list = aug_dataset.as_list(formatter)
                                if aug_list:
                                    aug_queries = [tokenizer.encode(s["text"]) for s in aug_list[:8]]
                                    aug_answers = [tokenizer.encode(s["reply"]) for s in aug_list[:8]]
                                    # Split into batches of 4
                                    half = max(1, len(aug_queries) // 2)
                                    s1 = calc_scores(aug_queries[:half], aug_answers[:half], tokenizer, model)
                                    s2 = calc_scores(aug_queries[half:half*2], aug_answers[half:half*2], tokenizer, model)
                                    aug_scores = s1 + s2
                                else:
                                    aug_scores = [10.0]
                                known_scores[(test_id, gh)] = aug_scores
                            all_candidates.append({
                                'test_id': test_id,
                                'grid': grid_list,
                                'beam_score': score,
                                'score_aug': aug_scores,
                                'is_dsl': False,
                                'dsl_program': None,
                            })
                        except Exception as e:
                            continue

            # ---------- Add DSL candidates (high confidence) ----------
            for dsl_out in dsl_candidates:
                gh = hashable(dsl_out)
                if (test_id, gh) not in known_scores:
                    # Score DSL candidate the same way (augmented NLL)
                    aug_dataset = ArcDataset(
                        keys=[test_id],
                        queries={test_id: data.queries[test_id]},
                        replies={test_id: [dsl_out]} if data.replies else {}
                    ).augment(seed=hash("dsl") % 1024**2)
                    aug_list = aug_dataset.as_list(formatter)
                    if aug_list:
                        aug_queries = [tokenizer.encode(s["text"]) for s in aug_list[:8]]
                        aug_answers = [tokenizer.encode(s["reply"]) for s in aug_list[:8]]
                        half = max(1, len(aug_queries) // 2)
                        s1 = calc_scores(aug_queries[:half], aug_answers[:half], tokenizer, model)
                        s2 = calc_scores(aug_queries[half:half*2], aug_answers[half:half*2], tokenizer, model)
                        aug_scores = s1 + s2
                    else:
                        aug_scores = [0.0]
                    known_scores[(test_id, gh)] = aug_scores
                else:
                    aug_scores = known_scores[(test_id, gh)]
                all_candidates.append({
                    'test_id': test_id,
                    'grid': dsl_out,
                    'beam_score': 0.0,  # DSL has no beam score; treat as best
                    'score_aug': aug_scores,
                    'is_dsl': True,
                    'dsl_program': None,  # not stored, but flagged
                })

            # ---------- Persist candidates ----------
            out_path = os.path.join(output_dir, f"{key}.bz2")
            with bz2.open(out_path, 'wb') as f:
                pickle.dump(all_candidates, f)

            elapsed = time.time() - puzzle_start
            print(f"[Rank {rank}] finished {key} in {elapsed:.1f}s "
                  f"({len(all_candidates)} candidates)")
            puzzles_processed += 1

        except Exception as e:
            import traceback
            print(f"[Rank {rank}] FAILED {key}: {e}")
            traceback.print_exc()
            # Don't re-raise — keep worker alive for next puzzle
            continue

    print(f"[Rank {rank}] worker exit after {puzzles_processed} puzzles")


Writing arc_solver.py


## Module 5: `arc_decoder.py` — Selection + Orthogonal 2-Attempt

**Selection algorithms (both kept for ablation):**
- `score_kgmon` — frequency − mean(min-NLL) (FIX: min-NLL, not mean-NLL)
- `score_full_probmul_3` — probability-mass style with min-NLL fix
- `reciprocal_rank_fusion` — combine both rankings (k=60)

**NEW: Orthogonal two-attempt selector** — `select_two_attempts()` decision tree:
1. If DSL candidate exists → attempt_1 = DSL, attempt_2 = top neural (different shape)
2. If only neural, ≥2 distinct grids → attempt_1 = top, attempt_2 = most structurally different
3. If only one neural candidate → attempt_1 = neural, attempt_2 = verified-transform or flip
4. If no candidates → attempt_1 = identity (test input), attempt_2 = vertical flip

**Diversity metric:** weighted combination of shape distance (0.4), color distance (0.3), transformation distance (0.3).


In [6]:
%%writefile arc_decoder.py
"""arc_decoder.py — Selection algorithms + orthogonal two-attempt diversifier.

Based on LB 33.89 baseline. Adds:
  - min-NLL instead of mean-NLL in score_kgmon and score_full_probmul_3
  - Orthogonal two-attempt selector: prefer verified symbolic + top neural
  - Identity/flip fallback instead of [[0]] (correct for ~13% of tasks)
"""
import os, json, bz2, pickle, math
import numpy as np
from collections import defaultdict
from arc_loader import hashable, is_valid_solution, identity_fallback, flip_fallback
from arc_verifier import verifier_ensemble, consistency_score_min_nll

# ---------- Selection algorithms ----------
def getter_full_probmul_3(guesses, baseline=3.0):
    """Probability-mass style: sum(baseline - beam) + mean(sum(baseline - aug)).
    FIX: uses min-NLL instead of mean-NLL across augmentations."""
    inf_score = float(np.sum([baseline - g["beam_score"] for g in guesses]))
    aug_score = float(np.mean([
        np.sum([baseline - s for s in [consistency_score_min_nll(g["score_aug"])]])
        for g in guesses
    ]))
    return inf_score + aug_score

def getter_kgmon(guesses):
    """Frequency style: count - mean(min-NLL).
    FIX: uses min-NLL instead of mean-NLL."""
    inf_score = len(guesses)
    aug_score = float(np.mean([consistency_score_min_nll(g["score_aug"]) for g in guesses]))
    return inf_score - aug_score

def score_sum(guesses, getter):
    """Deduplicate by grid content; aggregate scores of identical grids."""
    scores_dict = {}
    for g in guesses:
        h = hashable(g["grid"])
        if h not in scores_dict:
            scores_dict[h] = [[], g["grid"], g]
        scores_dict[h][0].append(g)
    # Compute aggregate score per unique grid
    scored = []
    for h, (group, grid, first_g) in scores_dict.items():
        s = getter(group)
        scored.append((s, grid, first_g))
    # Sort descending by score
    scored.sort(key=lambda x: x[0], reverse=True)
    return [(s, grid, g) for s, grid, g in scored]

# ---------- Reciprocal Rank Fusion ----------
def reciprocal_rank_fusion(rankings, k_const=60):
    """RRF: combine multiple rankings. Each ranking is a list of grids (best first).
    Returns a fused ranking."""
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, grid in enumerate(ranking):
            scores[hashable(grid)] += 1.0 / (k_const + rank + 1)
    # Sort by fused score descending
    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    # Map back to grids
    grid_lookup = {}
    for ranking in rankings:
        for grid in ranking:
            grid_lookup[hashable(grid)] = grid
    return [grid_lookup[h] for h, _ in fused if h in grid_lookup]

# ---------- Two-Attempt Diversifier (NEW) ----------
def diversity_score(g1, g2):
    """Structural diversity metric between two grids.
    Combines shape distance, color distance, and transformation distance.
    """
    a1 = np.array(g1) if g1 else np.array([[0]])
    a2 = np.array(g2) if g2 else np.array([[0]])
    # Shape distance
    h1, w1 = a1.shape
    h2, w2 = a2.shape
    shape_dist = abs(h1 - h2) + abs(w1 - w2)
    # Color distance (histogram L1)
    freq1 = np.bincount(a1.flatten(), minlength=10)
    freq2 = np.bincount(a2.flatten(), minlength=10)
    color_dist = np.abs(freq1 - freq2).sum()
    # Transformation distance (penalize simple transforms)
    trans_dist = 0
    if a1.shape == a2.shape:
        for name, fn in [("rot90", lambda x: np.rot90(x, 1)),
                         ("rot180", lambda x: np.rot90(x, 2)),
                         ("rot270", lambda x: np.rot90(x, 3)),
                         ("flip_h", lambda x: np.fliplr(x)),
                         ("flip_v", lambda x: np.flipud(x)),
                         ("transpose", lambda x: np.transpose(x))]:
            try:
                if np.array_equal(fn(a1), a2):
                    trans_dist = 1  # penalize transforms-of-each-other
                    break
            except Exception:
                continue
    return 0.4 * shape_dist + 0.3 * color_dist + 0.3 * trans_dist

def find_most_different(candidates, reference_grid):
    """Find the candidate grid most structurally different from reference."""
    if not candidates:
        return None
    best = None
    best_div = -1
    for c in candidates:
        g = c["grid"] if isinstance(c, dict) else c
        d = diversity_score(g, reference_grid)
        if d > best_div:
            best_div = d
            best = g
    return best or candidates[0].get("grid") if candidates else None

def select_two_attempts(candidates, train_pairs, test_input):
    """NEW: Orthogonal two-attempt selector.

    Decision tree (in order):
      1. If a DSL candidate exists -> attempt_1 = DSL, attempt_2 = top neural
      2. If only neural, >=2 distinct grids -> attempt_1 = top neural,
         attempt_2 = most structurally different neural
      3. If only one neural candidate -> attempt_1 = neural, attempt_2 = verified_transform
      4. If no candidates -> attempt_1 = identity, attempt_2 = vertical flip

    Returns (attempt_1, attempt_2) as grid arrays.
    """
    from arc_loader import verified_transform_solve

    # Separate DSL and neural candidates
    dsl_cands = [c for c in candidates if c.get("is_dsl", False)]
    neural_cands = [c for c in candidates if not c.get("is_dsl", False)]

    # Get unique grids
    seen = set()
    unique_neural = []
    for c in neural_cands:
        h = hashable(c["grid"])
        if h not in seen:
            seen.add(h)
            unique_neural.append(c)

    # Case 1: DSL candidate exists
    if dsl_cands:
        attempt_1 = dsl_cands[0]["grid"]
        if unique_neural:
            # Pick the top neural candidate if it differs from DSL
            top_neural = unique_neural[0]["grid"]
            if hashable(top_neural) != hashable(attempt_1):
                attempt_2 = top_neural
            else:
                attempt_2 = find_most_different(unique_neural, attempt_1)
        else:
            attempt_2 = find_most_different(unique_neural, attempt_1)
        if attempt_2 is None:
            # Fallback to verified transform
            vt = verified_transform_solve(train_pairs, test_input)
            attempt_2 = vt if vt is not None else flip_fallback(test_input)
        return attempt_1, attempt_2

    # Case 2: multiple distinct neural candidates
    if len(unique_neural) >= 2:
        attempt_1 = unique_neural[0]["grid"]
        attempt_2 = find_most_different(unique_neural, attempt_1)
        return attempt_1, attempt_2 or flip_fallback(test_input)

    # Case 3: only one neural candidate
    if unique_neural:
        attempt_1 = unique_neural[0]["grid"]
        # Try verified transform as attempt_2
        vt = verified_transform_solve(train_pairs, test_input)
        if vt is not None and hashable(vt) != hashable(attempt_1):
            attempt_2 = vt
        else:
            attempt_2 = flip_fallback(test_input)
        return attempt_1, attempt_2

    # Case 4: no candidates — last-resort fallbacks
    attempt_1 = identity_fallback(test_input)
    attempt_2 = flip_fallback(test_input)
    return attempt_1, attempt_2

# ---------- ArcDecoder ----------
class ArcDecoder:
    """Loads decoded candidate dumps, runs selection, produces submission-ready grids."""
    def __init__(self, dataset, n_guesses=2):
        self.dataset = dataset
        self.n_guesses = n_guesses

    @staticmethod
    def load_decoded_results(output_dir):
        """Load all .bz2 candidate dumps. Tolerates corrupt shards."""
        results = defaultdict(list)
        if not os.path.isdir(output_dir):
            return results
        for fname in os.listdir(output_dir):
            if not fname.endswith(".bz2"):
                continue
            try:
                with bz2.open(os.path.join(output_dir, fname), 'rb') as f:
                    cands = pickle.load(f)
                # fname is "{key}.bz2"
                key = fname[:-4]
                results[key] = cands
            except Exception as e:
                print(f"WARNING: corrupt shard {fname}: {e}")
                continue
        return results

    def run_selection_algo(self, all_candidates, getter=getter_kgmon):
        """Run selection algorithm on all candidates, grouped by test_id.
        Returns dict {test_id: [grid1, grid2]}."""
        results = {}
        # Group candidates by test_id
        by_test = defaultdict(list)
        for c in all_candidates:
            by_test[c["test_id"]].append(c)

        for test_id, cands in by_test.items():
            if not cands:
                continue
            # Run score_sum to dedupe + rank
            scored = score_sum(cands, getter)
            top_grids = [grid for _, grid, _ in scored[:self.n_guesses]]
            results[test_id] = top_grids
        return results

    def run_orthogonal_selection(self, all_candidates):
        """NEW: Run orthogonal two-attempt selector for each test_id."""
        results = {}
        by_test = defaultdict(list)
        for c in all_candidates:
            by_test[c["test_id"]].append(c)

        for test_id, cands in by_test.items():
            # Get train pairs and test input for this test_id
            base_key = test_id.split("_")[0] if "_" in test_id else test_id
            test_idx = int(test_id.split("_")[1]) if "_" in test_id else 0
            train_pairs, test_inputs = self.dataset.get_train_test(base_key)
            test_input = test_inputs[test_idx] if test_idx < len(test_inputs) else test_inputs[0]

            # Run orthogonal selector
            a1, a2 = select_two_attempts(cands, train_pairs, test_input.tolist())
            results[test_id] = [a1, a2]
        return results

    def benchmark_selection_algos(self, all_candidates, solutions):
        """Compare selection algorithms. Eval-mode only."""
        # Run each algo
        kgmon = self.run_selection_algo(all_candidates, getter_kgmon)
        probmul = self.run_selection_algo(all_candidates, getter_full_probmul_3)
        ortho = self.run_orthogonal_selection(all_candidates)

        # Compute accuracy
        def acc(results):
            correct = 0; total = 0
            for k, sols in solutions.items():
                for i, sol in enumerate(sols):
                    total += 1
                    test_id = f"{k}_{i}"
                    if test_id not in results:
                        # Try just the base key
                        if k in results and len(results[k]) > i:
                            guesses = results[k][:2]
                        else:
                            continue
                    else:
                        guesses = results[test_id][:2]
                    if not guesses:
                        continue
                    if hashable(guesses[0]) == hashable(sol):
                        correct += 1
                    elif len(guesses) > 1 and hashable(guesses[1]) == hashable(sol):
                        correct += 1
            return correct, total, correct / max(total, 1)

        for name, res in [("score_kgmon", kgmon),
                          ("score_full_probmul_3", probmul),
                          ("orthogonal_2attempt", ortho)]:
            c, t, a = acc(res)
            print(f"  {name}: {c}/{t} = {a:.3f}")
        return ortho  # return the orthogonal one as the chosen selection


Writing arc_decoder.py


## Module 6: `starter.py` — Multi-Process Harness

**4-GPU parallel processing** via `mp.spawn` with `Manager().Queue()`.

**Key features:**
- **Cheap-first task ordering** — sort by `estimate_work(train_tokens × 16 + test_tokens × 8 × n_test)` ascending; ensures tail is dropped under time pressure, not the cheap tasks
- **Sentinel shutdown** — 4 × `None` sentinels for clean exit
- **Per-task exception containment** — single bad task does not crash worker
- **OOM recovery** — `torch.cuda.synchronize()` + cache clear on `OutOfMemoryError`
- **Serial Unsloth patching** — file-marker handoff avoids disk contention during model load
- **PYTHONHASHSEED=0** — reproducibility across runs


In [7]:
%%writefile starter.py
"""starter.py — Multi-process harness: cheap-first ordering, sentinel shutdown.

Based on LB 33.89 starter.py. Adds:
  - Cheap-first task ordering via estimated_work
  - 13-min per-puzzle hard cap (was 20 min)
  - 6-min DFS sub-cap (was 9 min)
  - 4-min TTT explicit cap (NEW)
  - Per-task exception containment (preserved)
"""
import os, sys, time, argparse, multiprocessing as mp
import numpy as np

def estimate_work(key, queries):
    """Cheap-first ordering: estimate compute cost per puzzle.
    Smaller grids + fewer train pairs = cheaper.
    """
    q = queries[key]
    train_tokens = sum(len(p["input"]) * len(p["input"][0]) +
                       len(p["output"]) * len(p["output"][0])
                       for p in q["train"])
    # Median output/input ratio
    ratios = []
    for p in q["train"]:
        i_size = max(len(p["input"]) * len(p["input"][0]), 1)
        o_size = len(p["output"]) * len(p["output"][0])
        ratios.append(o_size / i_size)
    median_ratio = float(np.median(ratios)) if ratios else 1.0
    test_tokens = sum(len(t["input"]) * len(t["input"][0]) * (1 + median_ratio)
                      for t in q["test"])
    # Augmentation multiplier: 16 perms × 8 orientations × ntest
    n_test = len(q["test"])
    return train_tokens * 16 + test_tokens * 8 * n_test

def local_worker(rank, queue, end_time, model_path, formatter, test_path, output_dir, rerun_mode):
    """Per-GPU worker."""
    # Set CUDA device BEFORE importing arc_solver (which imports torch/unsloth)
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    os.environ["PYTHONHASHSEED"] = "0"

    # Set default device to CPU to avoid CUDA init in parent
    import torch
    torch.set_default_device("cpu")

    # Serialize Unsloth patching across ranks (avoid disk contention during model load)
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)

    from arc_solver import worker as solve_puzzle
    # Signal readiness
    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")

    # Run worker loop
    solve_puzzle(rank, queue, end_time, model_path, formatter, test_path, output_dir, rerun_mode)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, required=True,
                        help="Epoch timestamp for global end time")
    parser.add_argument("--test-path", type=str, default="/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
    parser.add_argument("--model-path", type=str, default="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1")
    parser.add_argument("--output-dir", type=str, default="/kaggle/inference_outputs")
    parser.add_argument("--n-procs", type=int, default=4)
    parser.add_argument("--rerun", action="store_true")
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)
    os.makedirs("/kaggle", exist_ok=True)

    # Load dataset
    from arc_loader import ArcDataset
    data = ArcDataset.from_file(args.test_path)

    # Cheap-first ordering: sort keys by estimated work ascending
    print("[starter] Computing cheap-first ordering...")
    work_estimates = [(k, estimate_work(k, data.queries)) for k in data.keys]
    work_estimates.sort(key=lambda x: x[1])  # ascending = cheapest first
    sorted_keys = [k for k, _ in work_estimates]
    print(f"[starter] {len(sorted_keys)} puzzles to process")
    print(f"[starter] Cheapest: {work_estimates[0]}")
    print(f"[starter] Most expensive: {work_estimates[-1]}")

    # Build queue
    ctx = mp.get_context('spawn')
    queue = ctx.Manager().Queue()
    for k in sorted_keys:
        queue.put(k)
    # Add sentinels (one per worker)
    for _ in range(args.n_procs):
        queue.put(None)

    # Load tokenizer for formatter (each worker will load its own model)
    print("[starter] Loading tokenizer for formatter...")
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(args.model_path, local_files_only=True)
    from arc_loader import QwenFormatter
    formatter = QwenFormatter(tokenizer, max_seq_length=8192)

    # Spawn workers
    print(f"[starter] Spawning {args.n_procs} workers, end_time={args.end_time}")
    ctx.spawn(
        local_worker,
        args=(queue, args.end_time, args.model_path, formatter, args.test_path, args.output_dir, args.rerun),
        nprocs=args.n_procs,
        join=True,
    )
    print("[starter] All workers exited")

if __name__ == "__main__":
    main()


Writing starter.py


## Run the Harness

Execute `starter.py` as a subprocess with the global end_time. The harness will:
1. Load the dataset and sort keys cheap-first
2. Spawn 4 worker processes (one per L4 GPU)
3. Each worker pulls puzzle keys from the shared queue
4. Workers process puzzles until queue empties or `end_time` is reached
5. Each puzzle's decoded candidates are persisted to `/kaggle/inference_outputs/{key}.bz2`

**Expected wall-clock:** ~11.5 hours for 240 tasks on 4× L4 (~13 min/puzzle × 60 puzzles/GPU).


In [8]:
# === CELL 16: RUN HARNESS ===
import subprocess, os

end_time_str = str(int(global_end_time))

print(f"=== Starting harness ===")
print(f"end_time = {end_time_str}")
print(f"test_path = {TEST_PATH}")
print(f"model_path = {MODEL_PATH}")
print(f"output_dir = {INFER_DIR}")
print(f"rerun_mode = {RERUN_MODE}")
print()

# Run starter.py as subprocess (preserves CUDA isolation per worker)
cmd = [
    'python', 'starter.py',
    '--end-time', end_time_str,
    '--test-path', TEST_PATH,
    '--model-path', MODEL_PATH,
    '--output-dir', INFER_DIR,
    '--n-procs', '4',
]
if RERUN_MODE:
    cmd.append('--rerun')

env = {**os.environ, 'PYTHONHASHSEED': '0'}
result = subprocess.run(cmd, env=env)

print(f"\n=== Harness exit code: {result.returncode} ===")
print(f"Inference outputs in: {INFER_DIR}")


=== Starting harness ===
end_time = 1789684287
test_path = /kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json
model_path = //kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1
output_dir = /kaggle/inference_outputs
rerun_mode = False

[starter] Computing cheap-first ordering...
[starter] 120 puzzles to process
[starter] Cheapest: ('e8686506', 8594.153846153846)
[starter] Most expensive: ('d8e07eb2', 147840.0)
[starter] Loading tokenizer for formatter...
[starter] Spawning 4 workers, end_time=1789684287.0


Traceback (most recent call last):
  File "/kaggle/working/starter.py", line 111, in <module>
    main()
  File "/kaggle/working/starter.py", line 102, in main
    ctx.spawn(
    ^^^^^^^^^
AttributeError: 'SpawnContext' object has no attribute 'spawn'



=== Harness exit code: 1 ===
Inference outputs in: /kaggle/inference_outputs


## Post-Process: Verifier Ensemble + Selection + Submission

Load all `.bz2` candidate dumps, run the verifier ensemble (min-NLL + functional),
select orthogonal two attempts per test, write `submission.json`.

**Submission schema validation:**
1. Every task_id in input is present
2. Every test has exactly 2 attempts
3. Every attempt is a rectangular grid of integers 0-9, 1×1 to 30×30
4. Identity/flip fallback for any missing attempts (better than `[[0]]`)


In [9]:
# === CELL 18: POST-PROCESS + WRITE SUBMISSION ===
import json, os, sys, time
import numpy as np

# Add CWD to path so we can import our modules
sys.path.insert(0, '/kaggle/working')

from arc_loader import ArcDataset, hashable, is_valid_solution, identity_fallback, flip_fallback, verified_transform_solve
from arc_decoder import ArcDecoder
from arc_verifier import verifier_ensemble, consistency_score_min_nll

# Load dataset
data = ArcDataset.from_file(TEST_PATH)
print(f"Loaded {len(data.keys)} tasks from {TEST_PATH}")

# Load all decoded candidates
all_results = ArcDecoder.load_decoded_results(INFER_DIR)
print(f"Loaded decoded results for {len(all_results)} tasks")

# Flatten all candidates
all_candidates = []
for key, cands in all_results.items():
    for c in cands:
        # Determine test_id from key (handles split_multi_replies format: key_0, key_1)
        if '_' in key and key.split('_')[0] in data.keys:
            base_key, idx = key.rsplit('_', 1)
            test_id = key
        else:
            test_id = key
        c['test_id'] = test_id
        all_candidates.append(c)
print(f"Total candidates: {len(all_candidates)}")

# Run orthogonal two-attempt selection
decoder = ArcDecoder(data, n_guesses=2)
print("Running orthogonal two-attempt selection...")
selected = decoder.run_orthogonal_selection(all_candidates)
print(f"Selected 2 attempts for {len(selected)} test_ids")

# Build submission
submission = {}
for key in data.keys:
    n_tests = len(data.queries[key]['test'])
    submission[key] = []
    for i in range(n_tests):
        test_id = f"{key}_{i}"
        if test_id in selected:
            a1, a2 = selected[test_id]
        elif key in selected and len(selected[key]) > i:
            a1, a2 = selected[key][0], selected[key][1] if len(selected[key]) > 1 else flip_fallback(data.queries[key]['test'][i]['input'])
        else:
            # Fallback: identity + flip (better than [[0]])
            test_in = data.queries[key]['test'][i]['input']
            a1 = identity_fallback(test_in)
            a2 = flip_fallback(test_in)
        # Ensure both are valid
        if not is_valid_solution(a1):
            a1 = identity_fallback(data.queries[key]['test'][i]['input'])
        if not is_valid_solution(a2):
            a2 = flip_fallback(data.queries[key]['test'][i]['input'])
        # Ensure both are int lists
        a1 = [[int(x) for x in row] for row in a1]
        a2 = [[int(x) for x in row] for row in a2]
        # Ensure distinct (if equal, replace attempt_2 with flip)
        if hashable(a1) == hashable(a2):
            a2 = flip_fallback(data.queries[key]['test'][i]['input'])
        submission[key].append({"attempt_1": a1, "attempt_2": a2})

# Validate schema
print("Validating submission schema...")
assert set(submission.keys()) == set(data.keys), "Keys mismatch!"
for k in data.keys:
    assert len(submission[k]) == len(data.queries[k]['test']), f"Wrong test count for {k}"
    for entry in submission[k]:
        assert "attempt_1" in entry and "attempt_2" in entry
        for a in ["attempt_1", "attempt_2"]:
            assert is_valid_solution(entry[a]), f"Invalid grid for {k}/{a}"
print("✓ Schema valid")

# Write submission.json
SUBMISSION_PATH = os.path.join(OUTPUT_DIR, 'submission.json')
with open(SUBMISSION_PATH, 'w') as f:
    json.dump(submission, f)

import hashlib
with open(SUBMISSION_PATH, 'rb') as f:
    sha = hashlib.sha256(f.read()).hexdigest()[:16]
print(f"\n✓ submission.json written to {SUBMISSION_PATH}")
print(f"  Size: {os.path.getsize(SUBMISSION_PATH)/1024:.1f} KB")
print(f"  Tasks: {len(submission)}")
print(f"  SHA-256 prefix: {sha}")


Loaded 120 tasks from /kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json
Loaded decoded results for 0 tasks
Total candidates: 0
Running orthogonal two-attempt selection...
Selected 2 attempts for 0 test_ids
Validating submission schema...
✓ Schema valid

✓ submission.json written to /kaggle/working/submission.json
  Size: 531.1 KB
  Tasks: 120
  SHA-256 prefix: a59823e2842b6423


## Validation (Eval Mode Only)

In commit/eval mode (not rerun), benchmark all three selection algorithms against
the evaluation solutions and print accuracy. This cell is skipped in rerun mode
because solutions are not available during scoring.


In [10]:
# === CELL 20: VALIDATION (eval mode only) ===
import json

if not RERUN_MODE and SOLN_PATH is not None and os.path.exists(SOLN_PATH):
    print("=== Benchmarking selection algorithms ===")
    with open(SOLN_PATH) as f:
        solutions = json.load(f)

    # Benchmark
    decoder.benchmark_selection_algos(all_candidates, solutions)

    # Final accuracy with orthogonal 2-attempt
    print("\n=== Final accuracy (orthogonal 2-attempt) ===")
    correct = 0; total = 0
    for k, sols in solutions.items():
        for i, sol in enumerate(sols):
            total += 1
            if k in submission and len(submission[k]) > i:
                a1 = submission[k][i]["attempt_1"]
                a2 = submission[k][i]["attempt_2"]
                if hashable(a1) == hashable(sol) or hashable(a2) == hashable(sol):
                    correct += 1
    if total > 0:
        acc = correct / total
        print(f"Accuracy: {correct}/{total} = {acc:.4f} ({acc*100:.2f}%)")
        print(f"Baseline (LB 33.89): 33.89%")
        print(f"Improvement: {(acc*100 - 33.89):+.2f} pp")
    else:
        print("No solutions found to validate against")
else:
    print("=== Rerun mode: skipping validation ===")
    print("Submission is ready. The competition server will score it.")
    print(f"submission.json: {os.path.join(OUTPUT_DIR, 'submission.json')}")

print("\n=== Done ===")


=== Benchmarking selection algorithms ===
  score_kgmon: 0/172 = 0.000
  score_full_probmul_3: 0/172 = 0.000
  orthogonal_2attempt: 0/172 = 0.000

=== Final accuracy (orthogonal 2-attempt) ===
Accuracy: 0/172 = 0.0000 (0.00%)
Baseline (LB 33.89): 33.89%
Improvement: -33.89 pp

=== Done ===
